# 🏏 Indian Cricket Team Member Face Identification
### End-to-End Computer Vision & Machine Learning Pipeline

**Objective:**
Develop a robust Machine Learning & Computer Vision system to detect and identify Indian Cricket Team players (e.g. *Virat Kohli, Rohit Sharma, MS Dhoni, Sachin Tendulkar, Jasprit Bumrah, Hardik Pandya, Ravindra Jadeja, Shubman Gill, KL Rahul, Rishabh Pant*) from images.

**Pipeline Overview:**
1. **Face Localization & Alignment:** OpenCV Haar Cascade multi-scale face detection.
2. **Feature Engineering:** 2D Discrete Wavelet Transform (DWT) edge frequency extraction + Raw pixel intensity fusion.
3. **Dimensionality Reduction:** `StandardScaler` + Principal Component Analysis (`PCA`).
4. **Classification:** Support Vector Machine (`SVM`) with RBF kernel and `GridSearchCV` hyperparameter tuning.
5. **Evaluation:** Confusion Matrix, Precision, Recall, F1-Score, and Real-time Inference.

In [ ]:
import os
import sys
import json
import joblib
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set dark plot styling
plt.style.use('dark_background')
print("[✓] Libraries successfully imported!")

## 1. Dataset Initialization & Face Preprocessing

In [ ]:
# Add root dir to sys path
sys.path.append('..')
from src.config import INDIAN_CRICKETERS, PLAYER_DISPLAY_NAMES, CROPPED_DATA_DIR
from src.dataset_downloader import DatasetManager

# Initialize starter dataset for all 10 players
dm = DatasetManager()
stats = dm.generate_starter_dataset(samples_per_player=25)
print("Dataset summary:", stats)

## 2. Visualizing 2D Wavelet Transformation
2D Wavelet transform strips high-frequency illumination noise and extracts facial bone structure, eyes, nose, and lips.

In [ ]:
from src.preprocessor import FeatureExtractor

extractor = FeatureExtractor()
sample_path = list(CROPPED_DATA_DIR.glob('*/*.jpg'))[0]
sample_bgr = cv2.imread(str(sample_path))
sample_rgb = cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2RGB)
wavelet_img = extractor.extract_wavelet_image(sample_bgr)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample_rgb)
axes[0].set_title("Original Face Crop (160x160)", color='#00d2d3')
axes[0].axis('off')

axes[1].imshow(wavelet_img, cmap='gray')
axes[1].set_title("2D Wavelet Edge Frequency Features", color='#ffd166')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 3. Feature Extraction & Dataset Matrix Construction

In [ ]:
from src.train import ModelTrainer

trainer = ModelTrainer()
X, y, class_dict = trainer.load_dataset_features()
print(f"Feature matrix shape: {X.shape}")
print(f"Label vector shape: {y.shape}")
print(f"Classes: {class_dict}")

## 4. Train/Test Split & Principal Component Analysis (PCA)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
pca = PCA().fit(X_train_scaled)

plt.figure(figsize=(8, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_), color='#00f0ff', lw=2)
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Explained Variance Ratio', color='#00d2d3')
plt.grid(alpha=0.2)
plt.show()

## 5. Model Training & GridSearchCV Benchmark

In [ ]:
benchmark = trainer.train_and_tune(test_size=0.20)
print(f"[🏆] Winner Model: {benchmark['best_model_name']} with {benchmark['best_accuracy']*100:.2f}% Test Accuracy!")

## 6. Model Evaluation & Confusion Matrix Heatmap

In [ ]:
from src.evaluate import ModelEvaluator

evaluator = ModelEvaluator()
metrics = evaluator.evaluate_model()
print(f"Overall Evaluation Accuracy: {metrics['overall_accuracy']*100:.2f}%")

## 7. Single Image Test & Player Profile Recognition HUD

In [ ]:
from src.predict import FacePredictor

predictor = FacePredictor()
test_image_path = list(CROPPED_DATA_DIR.glob('*/*.jpg'))[5]
results = predictor.predict_image(test_image_path)

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(results['annotated_image'], cv2.COLOR_BGR2RGB))
plt.title(f"Identified: {results['detections'][0]['player_name']} ({int(results['detections'][0]['confidence']*100)}%)", color='#00d2d3')
plt.axis('off')
plt.show()